In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))
sys.path.append(str(PROJECT_ROOT / "fine_tune"))

In [4]:
import importlib
import config
import training.embedding_trainer
importlib.reload(config)
importlib.reload(training.embedding_trainer)

from config import (
    FINETUNED_MODEL_DIR,
    FINETUNE_MODEL_NAME,
    CATEGORY_COLUMN,
    TRAIN_BATCH_SIZE,
    EPOCHS,
    LEARNING_RATE,
    WEIGHT_DECAY,
    MAX_SEQ_LENGTH,
)

In [4]:
import shutil
shutil.rmtree(str(FINETUNED_MODEL_DIR), ignore_errors=True)

In [5]:
import sentence_transformers
from training.dataset_builder import load_finetune_data
from training.embedding_trainer import build_sentence_transformer
from training.evaluator import KNNCategoryEvaluator
from training.embedding_trainer import train
import time
import os
from sentence_transformers import SentenceTransformer


In [6]:
print(f"Base model:        {FINETUNE_MODEL_NAME}")
print(f"Output directory:  {FINETUNED_MODEL_DIR}")
print(f"Batch size:        {TRAIN_BATCH_SIZE}")
print(f"Epochs:            {EPOCHS}")
print(f"Learning rate:     {LEARNING_RATE}")
print(f"Weight decay:      {WEIGHT_DECAY}")
print(f"Max seq length:    {MAX_SEQ_LENGTH}")

Base model:        HooshvareLab/bert-base-parsbert-uncased
Output directory:  D:\proposal\thiese\code\Persian-Topic-Modeling\models\finetuned\batch_hard_triplet
Batch size:        16
Epochs:            8
Learning rate:     2e-05
Weight decay:      0.01
Max seq length:    256


In [7]:
print(f"sentence-transformers version: {sentence_transformers.__version__}")

assert sentence_transformers.__version__.split(".")[0] >= "3", (
    "This notebook uses the SentenceTransformerTrainer API (>=3.0). "
    "Installed version is older -- code will not work as-is."
)

sentence-transformers version: 5.1.1


In [8]:
# --------------------------------------------------
# Step 1: Load and prepare fine-tuning data
# --------------------------------------------------
# Only train.csv is used here. test.csv is never touched until
# 03_comparison.ipynb, so the final NMI/purity/coherence comparison
# against baseline stays leakage-free.


train_dataset, val_texts, val_labels, label_encoder = load_finetune_data(
    label_column=CATEGORY_COLUMN,
    val_size=0.1,
)

print(train_dataset)
print(f"\nExample row: {train_dataset[0]}")
print(f"\nLabel classes: {list(label_encoder.classes_)}")

Fine-tune train examples: 4297
Fine-tune val examples:   478
Number of classes:        5
Dataset({
    features: ['sentence', 'label'],
    num_rows: 4297
})

Example row: {'sentence': 'به گزارش ایسنا بیتکوین پنجم فوریه ۱۶ بهمن پس از سقوط حدود ۵۰ درصدی از رکورد ۱۲۶ هزار دلار در اکتبر گذشته به محدوده ۶۰ هزار دلار رسید بخش تحقیقاتی صرافی بایننس اعلام کرد که این کاهش در مقایسه با چرخههای گذشته بازار به جای سفتهبازیهای خرد بیشتر تحت تاثیر سرمایههای نهادی و عوامل کلان شکلگرفته است به گفته تحلیلگران بایننس عقبنشینی ۵۰ درصدی فعلی اصلاحی متوسط نسبت به چرخههای قبلی محسوب میشود و تاکنون بیتکوین ۹ افت قیمت جداگانه با این بزرگی یا بزرگتر را ثبت کرده است از جمله نمونههای تاریخی میتوان به دو سقوط ۹۴ درصدی در سالهای ۲۰۱۰ و ۲۰۱۱ افت ۷۸ درصدی بین نوامبر ۲۰۲۱ تا نوامبر ۲۰۲۲ و ریزش ۸۴ درصدی در بازار نزولی ۲۰۱۷ تا ۲۰۱۸ اشاره کرد این صرافی در گزارش خود تاکید کرد که افت اخیر بیشتر تحت تاثیر شرایط کلان اقتصادی و نه بحران بازار رمزارزها رخداده است دادههای مطلوب بازار کار آمریکا و ابهام درباره سیاستهای فدرال ر

In [9]:
# --------------------------------------------------
# Step 2: Build the base sentence-embedding model
# --------------------------------------------------
# ParsBERT is a raw HF encoder, not a native sentence-transformers model --
# this wraps it with a mean-pooling head so .encode() produces sentence
# vectors, not per-token vectors.

model = build_sentence_transformer(
    model_name=FINETUNE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


In [10]:
# --------------------------------------------------
# Step 3: Build the validation evaluator
# --------------------------------------------------
# After each eval_steps interval, this embeds val_texts and checks how
# accurately a simple k-NN classifier can predict category from the
# embeddings alone. Rising accuracy = fine-tuning is separating
# categories better. Purely for checkpoint selection -- not the final
# thesis metric (that's evaluate_all() in 03_comparison.ipynb, on test.csv).

evaluator = KNNCategoryEvaluator(val_texts, val_labels, name="val", k=5)

# Quick sanity check before training: baseline (untrained) separation
baseline_score = evaluator(model, epoch=-1, steps=-1)
print(f"Pre-finetuning val k-NN accuracy: {baseline_score:.4f}")

[epoch -1] KNN(k=5) val accuracy: 0.7992
Pre-finetuning val k-NN accuracy: 0.7992


In [11]:
# --------------------------------------------------
# Step 4: Fine-tune with supervised batch-hard triplet loss
# --------------------------------------------------
# BatchSamplers.GROUP_BY_LABEL (inside embedding_trainer.build_training_args)
# ensures each batch contains multiple documents per category, so the loss
# can mine hardest-positive / hardest-negative pairs within every batch.

FINETUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

start_time = time.time()

model = train(
    model=model,
    train_dataset=train_dataset,
    evaluator=evaluator,
    output_dir=str(FINETUNED_MODEL_DIR),
)

end_time = time.time()
print(f"\nFine-tuning took {(end_time - start_time)/60:.1f} minutes")

Step,Training Loss,Validation Loss,Evaluator
100,5.068300,No log,0.824211
200,5.002900,No log,0.841096
300,5.010700,No log,0.826360
400,5.034800,No log,0.853575
500,5.016700,No log,0.901732
600,5.008900,No log,0.905899
700,4.956000,No log,0.910132
800,4.950000,No log,0.914276
900,4.937700,No log,0.899627
1000,5.000000,No log,0.901732


[epoch 0.7434944237918215] KNN(k=5) val accuracy: 0.8242
[epoch 1.483271375464684] KNN(k=5) val accuracy: 0.8411
[epoch 2.2230483271375463] KNN(k=5) val accuracy: 0.8264
[epoch 2.966542750929368] KNN(k=5) val accuracy: 0.8536
[epoch 3.7063197026022303] KNN(k=5) val accuracy: 0.9017
[epoch 4.446096654275093] KNN(k=5) val accuracy: 0.9059
[epoch 5.185873605947956] KNN(k=5) val accuracy: 0.9101
[epoch 5.929368029739777] KNN(k=5) val accuracy: 0.9143
[epoch 6.669144981412639] KNN(k=5) val accuracy: 0.8996
[epoch 7.408921933085502] KNN(k=5) val accuracy: 0.9017
Fine-tuned model saved to: D:\proposal\thiese\code\Persian-Topic-Modeling\models\finetuned\batch_hard_triplet

Fine-tuning took 48.8 minutes


In [12]:
# --------------------------------------------------
# Step 5: Confirm the model was saved correctly
# --------------------------------------------------

saved_files = os.listdir(FINETUNED_MODEL_DIR)
print(f"Files in {FINETUNED_MODEL_DIR}:")
for f in saved_files:
    print(f"  {f}")

assert "config.json" in saved_files or "modules.json" in saved_files, (
    "Expected sentence-transformers model files not found -- save may have failed."
)

Files in D:\proposal\thiese\code\Persian-Topic-Modeling\models\finetuned\batch_hard_triplet:
  1_Pooling
  checkpoint-1000
  checkpoint-1080
  config.json
  config_sentence_transformers.json
  eval
  model.safetensors
  modules.json
  README.md
  sentence_bert_config.json
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json
  vocab.txt


In [13]:
# --------------------------------------------------
# Step 6: Reload the saved model and verify it loads cleanly
# --------------------------------------------------
# This is the exact loading pattern 03_comparison.ipynb will use.

finetuned_model = SentenceTransformer(str(FINETUNED_MODEL_DIR))

final_score = evaluator(finetuned_model, epoch="final", steps=-1)
print(f"Post-finetuning val k-NN accuracy: {final_score:.4f}")
print(f"Improvement over baseline: {final_score - baseline_score:+.4f}")

[epoch final] KNN(k=5) val accuracy: 0.9017
Post-finetuning val k-NN accuracy: 0.9017
Improvement over baseline: +0.1026


In [3]:
# ====================================================
# PART 2: Build BERTopic with the fine-tuned embedding
# model, fit on train, evaluate on test
# (mirrors 01_baseline_bertopic.ipynb structure exactly,
#  only the embedding model differs)
# ====================================================

import pandas as pd
import numpy as np
import time
from IPython.display import display, Markdown

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

from config import (
    TRAIN_DATA_PATH, TEST_DATA_PATH, TABLES_DIR, FIGURES_DIR,
    FINETUNED_MODEL_DIR, N_NEIGHBORS, N_COMPONENTS, MIN_DIST, UMAP_METRIC,
    RANDOM_STATE, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_METHOD,
    PREDICTION_DATA, NGRAM_RANGE, CALCULATE_PROBABILITIES, VERBOSE,
    LANGUAGE, BATCH_SIZE, MAX_LENGTH
)
from preprocessing import persian_stopwords
from evaluation import TopicEvaluator
from statistic import TopicStatistics
from visualizer import TopicVisualizer

In [18]:
def show_table( table_name):
    """
    Display a generated statistics table.
    """
    table = pd.read_csv(
        TABLES_DIR / "finetuned_tables" / f"{table_name}.csv"
    )

    display(
        Markdown(
            f"## {table_name.replace('_', ' ').title()}"
        )
    )

    display(
        table.style.hide(axis="index")
    )

In [5]:
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)

train_documents = train_df['content'].astype(str).tolist()
test_documents = test_df['content'].astype(str).tolist()

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 4775
Test size: 1194


In [6]:
# Load the checkpoint saved at the end of Part 1 training
finetuned_embedding_model = SentenceTransformer(str(FINETUNED_MODEL_DIR), device='cuda')
finetuned_embedding_model.max_seq_length = MAX_LENGTH

print(f"✅ Fine-tuned model loaded from {FINETUNED_MODEL_DIR}")

✅ Fine-tuned model loaded from D:\proposal\thiese\code\Persian-Topic-Modeling\models\finetuned\batch_hard_triplet


In [7]:
# Same UMAP / HDBSCAN / vectorizer config as baseline -- this is what
# keeps the comparison fair. Only the embedding model changes.

umap_model = UMAP(
    n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
    min_dist=MIN_DIST, metric=UMAP_METRIC,
    random_state=RANDOM_STATE, low_memory=False
)

cluster_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
    metric='euclidean', cluster_selection_method=CLUSTER_SELECTION_METHOD,
    prediction_data=PREDICTION_DATA
)

vectorizer_model = CountVectorizer(
    stop_words=list(persian_stopwords), ngram_range=NGRAM_RANGE, min_df=5
)

finetuned_topic_model = BERTopic(
    embedding_model=finetuned_embedding_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    language=LANGUAGE,
    calculate_probabilities=CALCULATE_PROBABILITIES,
    verbose=VERBOSE
)

In [8]:
train_embeddings = finetuned_embedding_model.encode(
    train_documents, batch_size=BATCH_SIZE, show_progress_bar=True
)

start_time = time.time()
topics, probabilities = finetuned_topic_model.fit_transform(train_documents, train_embeddings)
end_time = time.time()

print(f"Training BERTopic (fine-tuned embeddings): {end_time - start_time:.2f} seconds")

Batches: 100%|██████████| 299/299 [01:46<00:00,  2.80it/s]
2026-08-15 02:35:46,988 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-15 02:36:10,199 - BERTopic - Dimensionality - Completed ✓
2026-08-15 02:36:10,200 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-15 02:36:10,376 - BERTopic - Cluster - Completed ✓
2026-08-15 02:36:10,380 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-15 02:36:12,593 - BERTopic - Representation - Completed ✓


Training BERTopic (fine-tuned embeddings): 26.50 seconds


In [9]:
hierarchical_topics = finetuned_topic_model.hierarchical_topics(train_documents)
topic_tree = finetuned_topic_model.get_topic_tree(hierarchical_topics)
print(topic_tree)

100%|██████████| 9/9 [00:00<00:00, 345.94it/s]

.
├─تیم_ایران_بازی_تیم ملی_کیلوگرم
│    ├─■──کیلوگرم_وزن_کشتی_مدال_مسابقات ── Topic: 3
│    └─تیم_بازی_تیم ملی_ایران_ملی
│         ├─■──والیبال_تیم_۲۵_تیم ملی_ایران ── Topic: 8
│         └─■──تیم_بازی_فوتبال_بازیکنان_لیگ ── Topic: 2
└─کشور_ایران_قرار_اسلامی_جنگ
     ├─کشور_ایران_اسلامی_قرار_جنگ
     │    ├─■──آموزش_پرورش_آموزش پرورش_مدارس_دانشآموزان ── Topic: 6
     │    └─ایران_کشور_اسلامی_جنگ_قرار
     │         ├─■──سلامت_بهداشت_پزشکی_دارو_بیماری ── Topic: 4
     │         └─ایران_کشور_اسلامی_جنگ_قرار
     │              ├─■──کشور_ایران_اسلامی_جنگ_انقلاب ── Topic: 0
     │              └─■──کتاب_آثار_ایران_موسیقی_نمایش ── Topic: 1
     └─درصد_برق_دلار_افزایش_کاهش
          ├─■──برق_آب_مصرف_انرژی_کشور ── Topic: 7
          └─درصد_دلار_افزایش_نفت_کاهش
               ├─■──نفت_بشکه_انرژی_افزایش_دلار ── Topic: 9
               └─■──درصد_دلار_قیمتی_تغییرات_کاهش ── Topic: 5



In [10]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
hierarchical_topics.to_csv(
    TABLES_DIR / "finetuned_hierarchical_topics.csv",
    index=False, encoding='utf-8-sig'
)

In [11]:
test_embeddings = finetuned_embedding_model.encode(
    test_documents, batch_size=BATCH_SIZE, show_progress_bar=True
)

start = time.time()
test_topics, test_probabilities = finetuned_topic_model.transform(test_documents, test_embeddings)
end = time.time()

print(f"Transform Test: {end - start:.2f} seconds")
print(f"Outliers: {(np.array(test_topics)==-1).sum()}/{len(test_topics)}")

Batches: 100%|██████████| 75/75 [00:25<00:00,  2.90it/s]
2026-08-15 02:39:25,004 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-08-15 02:39:34,197 - BERTopic - Dimensionality - Completed ✓
2026-08-15 02:39:34,197 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-08-15 02:39:34,224 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-08-15 02:39:34,303 - BERTopic - Probabilities - Completed ✓
2026-08-15 02:39:34,304 - BERTopic - Cluster - Completed ✓


Transform Test: 9.30 seconds
Outliers: 40/1194


In [12]:
# --------------------------------------------------
# Evaluate on TEST data — same 5 metrics as baseline
# --------------------------------------------------

topic_evaluator = TopicEvaluator(
    topic_model=finetuned_topic_model,
    documents=test_documents,
    topics=test_topics,
    probabilities=test_probabilities,
    dataframe=test_df
)

results = topic_evaluator.evaluate_all()
topic_evaluator.save_results(TABLES_DIR / "finetuned_topic_evaluation.csv")

results

Evaluation results saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_topic_evaluation.csv


,Metric,Value
0,Coherence (using c_v methode),0.7364
1,Diversity,0.8100
2,Outlier Ratio (%),3.3501
3,NMI,0.7040
4,Purity,0.9003


In [13]:
# --------------------------------------------------
# Topic statistics + tables (fit-side, same pattern as baseline)
# --------------------------------------------------

topic_statistics = TopicStatistics(
    topic_model=finetuned_topic_model,
    documents=train_documents,
    topics=topics,
    dataframe=train_df,
    probabilities=probabilities
)

topic_tables = topic_statistics.generate_all_tables()
topic_statistics.save_all_tables(TABLES_DIR / "finetuned_tables")

Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_statistics.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_information.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_keywords.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_sizes.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_percentages.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\representative_documents.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\average_document_length_per_topic.csv
Saved successfully:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\tables\finetuned_tables\topic_category_distribution.csv
Saved suc

In [14]:
# --------------------------------------------------
# Figures — non-comparison mode (compration_mode=False)
# generates: topic map, barchart, hierarchy, heatmap, document map
# --------------------------------------------------

topic_visualizer = TopicVisualizer(
    topic_model=finetuned_topic_model,
    statistics=topic_statistics
)

topic_figs = topic_visualizer.generate_all_figs(
    document=train_documents,
    embeding=train_embeddings,
    compration_mode=False
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
topic_visualizer.save_all_figures(FIGURES_DIR / "finetuned_figs", as_html=True)


All figures were saved to:
D:\proposal\thiese\code\Persian-Topic-Modeling\results\figures\finetuned_figs


In [15]:
n_docs = len(train_documents)
n_topics = len(finetuned_topic_model.get_topic_info()) - 1
n_outliers = np.sum(np.array(topics) == -1)

print(f"Number of documents : {n_docs}")
print(f"Number of topics    : {n_topics}")
print(f"Outlier documents   : {n_outliers}")
print(f"Outlier percentage  : {100*n_outliers/n_docs:.2f}%")

Number of documents : 4775
Number of topics    : 10
Outlier documents   : 165
Outlier percentage  : 3.46%


In [19]:
for table in topic_tables.keys():
    if table !='topic_information':
        print("**********************")
        print(f"⭕ {table}")
        print("**********************")
        show_table( table)

**********************
⭕ topic_statistics
**********************


## Topic Statistics

Statistic,Value
Number of Topics,10.000000
Average Documents per Topic,461.000000
Largest Topic,1248.000000
Smallest Topic,141.000000
Outlier Documents,165.000000
Outlier Percentage (%),3.460000


**********************
⭕ topic_keywords
**********************


## Topic Keywords

...

## Topic Category Distribution

Topic,category,Count,Percentage
0,اجتماعی,12,0.960000
0,اقتصادی,293,23.480000
0,سیاسی,926,74.200000
0,فرهنگی و هنری,17,1.360000
1,اجتماعی,11,1.180000
1,اقتصادی,5,0.530000
1,سیاسی,9,0.960000
1,فرهنگی و هنری,904,96.680000
1,ورزشی,6,0.640000
2,سیاسی,2,0.420000


**********************
⭕ topic_subcategory_distribution
**********************


## Topic Subcategory Distribution

Topic,sub_category,Count,Percentage
0,آموزش و پرورش,10,0.800000
0,ادبیات و کتاب,4,0.320000
0,ارتباطات و فناوری اطلاعات,77,6.170000
0,اقتصاد کلان,198,15.870000
0,انرژی,18,1.440000
0,تجسمی و موسیقی,5,0.400000
0,حقوقی و قضایی,298,23.880000
0,دفاعی - امنیتی,263,21.070000
0,سلامت,2,0.160000
0,سیاست داخلی,365,29.250000


In [20]:
fig = finetuned_topic_model.visualize_topics()

fig.show()

In [22]:
fig = finetuned_topic_model.visualize_barchart(
    top_n_topics=20
)

fig.show()

In [23]:
fig = finetuned_topic_model.visualize_heatmap()

fig.show()

In [24]:
fig = finetuned_topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)

fig.show()